# Global rotations of the Steane code

This tutorial reproduces the two-dimensional color-code part of Fig. 4a from [Bluvstein et al., *Nature* 649, 39-46 (2026)](https://www.nature.com/articles/s41586-025-09848-5) with Tsim. The experiment prepares a small error-correcting code, applies the same physical phase rotation to every data qubit, and then asks which rotation angles preserve the code space and the logical observable.

The `[[7, 1, 3]]` Steane code is the required 2D color-code curve from the unitaryHACK issue. The `[[15, 1, 3]]` Reed-Muller curve is not included here because it requires many simultaneous non-Clifford rotations and the issue marks it as optional when it is not practical to run quickly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from tsim import Circuit

## Encode a logical qubit

The Steane encoder below is the deterministic `[[7, 1, 3]]` circuit already used in the Tsim color-code demo. It prepares a logical `|+_L>` state. We measure six stabilizer generators and one logical `X_L` operator after applying the global rotation.

One X-type stabilizer is written with a leading `!` because this particular deterministic encoder prepares that generator with negative sign. The inversion normalizes all six detector columns so that a zero detector bit means "the expected stabilizer value was recovered."

In [ ]:
STEANE_ENCODER = """
RX 6
R 0 1 2 3 4 5
SQRT_Y_DAG 0 1 2 3 4 5
CZ 1 2 3 4 5 6
SQRT_Y 6
CZ 0 3 2 5 4 6
SQRT_Y 2 3 4 5 6
CZ 0 1 2 3 4 5
SQRT_Y 1 2 4
X 3
"""

X_STABILIZERS = [
    "!X0*X1*X2*X3",
    "X1*X2*X4*X5",
    "X2*X3*X4*X6",
]
Z_STABILIZERS = [
    "Z0*Z1*Z2*Z3",
    "Z1*Z2*Z4*Z5",
    "Z2*Z3*Z4*Z6",
]
LOGICAL_X = "X0*X1*X5"

In [ ]:
def make_steane_rotation_circuit(theta_over_pi: float) -> Circuit:
    pauli_measurements = " ".join(X_STABILIZERS + Z_STABILIZERS + [LOGICAL_X])
    return Circuit(f"""
{STEANE_ENCODER}
R_Z({theta_over_pi}) 0 1 2 3 4 5 6
MPP {pauli_measurements}
DETECTOR rec[-7]
DETECTOR rec[-6]
DETECTOR rec[-5]
DETECTOR rec[-4]
DETECTOR rec[-3]
DETECTOR rec[-2]
OBSERVABLE_INCLUDE(0) rec[-1]
""")

circuit = make_steane_rotation_circuit(0.25)
print(f"Qubits: {circuit.num_qubits}")
print(f"Detectors: {circuit.num_detectors}")
print(f"Observables: {circuit.num_observables}")

The global `R_Z(theta)` instruction rotates every physical data qubit by `theta * pi`. Tsim then samples the stabilizer detectors and logical observable directly from the non-Clifford circuit.

In [ ]:
circuit.diagram("timeline-svg", height=360)

## Sweep the rotation angle

For each angle, we estimate three quantities:

- the mean stabilizer expectation, averaged over the six measured generators;
- the raw logical `X_L` expectation;
- the postselected logical `X_L` expectation using only shots with all stabilizers restored.

The last column is useful as a tutorial proxy for the postselection used in the paper: it shows the logical response conditioned on returning to the expected code space.

In [ ]:
def bit_expectation(bits: np.ndarray) -> np.ndarray:
    return 1 - 2 * bits.mean(axis=0)


def evaluate_steane(theta_over_pi: float, shots: int = 1024) -> dict[str, float]:
    sampler = make_steane_rotation_circuit(theta_over_pi).compile_detector_sampler(seed=1234)
    detectors, observables = sampler.sample(
        shots=shots,
        batch_size=shots,
        separate_observables=True,
    )
    postselection_mask = np.all(detectors == 0, axis=1)
    if np.any(postselection_mask):
        postselected_logical = bit_expectation(observables[postselection_mask, 0])[()]
    else:
        postselected_logical = np.nan

    return {
        "theta_over_pi": theta_over_pi,
        "stabilizer": float(bit_expectation(detectors).mean()),
        "logical_x": float(bit_expectation(observables[:, 0])),
        "postselected_logical_x": float(postselected_logical),
        "acceptance": float(postselection_mask.mean()),
    }


angles = np.linspace(0, 1, 17)
results = [evaluate_steane(float(theta)) for theta in angles]
results[:3]

## Add an unentangled reference

Fig. 4a also compares encoded data to an unentangled experiment. Here we prepare seven independent `|+>` qubits, apply the same global rotation, and measure the same weight-three `X` parity used as the logical reference. It is not protected by stabilizers, so it changes smoothly with the physical rotation angle.

In [ ]:
def make_unentangled_reference(theta_over_pi: float) -> Circuit:
    return Circuit(f"""
RX 0 1 2 3 4 5 6
R_Z({theta_over_pi}) 0 1 2 3 4 5 6
MPP X0*X1*X5
OBSERVABLE_INCLUDE(0) rec[-1]
""")


def evaluate_reference(theta_over_pi: float, shots: int = 1024) -> float:
    sampler = make_unentangled_reference(theta_over_pi).compile_detector_sampler(seed=5678)
    _, observables = sampler.sample(
        shots=shots,
        batch_size=shots,
        separate_observables=True,
    )
    return float(bit_expectation(observables[:, 0]))


reference_logical = np.array([evaluate_reference(float(theta)) for theta in angles])

## Plot the reproduced curve

The encoded stabilizers revive at Clifford-protected angles, while the unentangled reference lacks those code-space revivals. The postselected logical curve highlights the plateau behavior after keeping only shots whose stabilizers match the expected signs.

In [ ]:
theta_degrees = angles * 180
stabilizer = np.array([row["stabilizer"] for row in results])
logical_x = np.array([row["logical_x"] for row in results])
postselected_logical_x = np.array([row["postselected_logical_x"] for row in results])
acceptance = np.array([row["acceptance"] for row in results])

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(theta_degrees, stabilizer, "o-", label="Steane stabilizers")
ax.plot(theta_degrees, logical_x, "o-", label="Steane logical $X_L$")
ax.plot(theta_degrees, postselected_logical_x, "o-", label="Postselected $X_L$")
ax.plot(theta_degrees, reference_logical, "--", label="Unentangled reference")

for angle in (0, 90, 180):
    ax.axvline(angle, color="0.85", lw=0.8, zorder=0)

ax.set_xlabel(r"Global phase rotation $	heta$ (degrees)")
ax.set_ylabel("Expectation value")
ax.set_ylim(-1.05, 1.05)
ax.set_xticks([0, 45, 90, 135, 180])
ax.legend(loc="best")
ax.grid(alpha=0.25)
fig.tight_layout();

The acceptance fraction peaks when the global rotation maps the code space back onto itself. Away from those angles, stabilizer measurements detect that the state has leaked out of the expected code sector more often.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.8))
ax.plot(theta_degrees, acceptance, "o-", color="tab:green")
ax.set_xlabel(r"Global phase rotation $	heta$ (degrees)")
ax.set_ylabel("Code-space acceptance")
ax.set_ylim(0, 1.05)
ax.set_xticks([0, 45, 90, 135, 180])
ax.grid(alpha=0.25)
fig.tight_layout();

## What Tsim computed

Tsim compiled the non-Clifford global rotations and sampled detector/observable outcomes from the resulting Clifford+rotation circuit. The stabilizer detectors are parity checks that indicate whether the encoded state returned to the expected Steane code sector. The logical observable estimates the encoded qubit response to the global physical rotation.

This is the same mechanism emphasized in Fig. 4a: encoded states respond differently from unentangled physical qubits because the stabilizer structure filters which physical rotations act like robust logical operations.